# Getting started with DonorData

This tutorial builds a small `DonorData` from scratch and shows its levels, selections, how information moves
between levels, and how the object is written to disk. The data is simulated.

In [1]:
import numpy as np
import pandas as pd
from anndata import AnnData

import donordata
from donordata import DonorData

rng = np.random.default_rng(0)

## Building a DonorData

A `DonorData` is built from three parts. The donor table has one row per donor, the sample table one row per sample
with the donor of every sample, and the cells are an AnnData with the sample of every cell in `obs`. The sample table
is optional; without it, `C.obs` names the donor of every cell.

In [2]:
donors = pd.DataFrame(
    {"age": [34, 52, 61, 70], "disease": ["healthy", "healthy", "covid", "covid"]},
    index=pd.Index(["P1", "P2", "P3", "P4"], name="donor_id"),
)
samples = pd.DataFrame(
    {"donor_id": ["P1", "P2", "P3", "P3", "P4", "P4"], "day": [0, 0, 0, 7, 0, 7]},
    index=pd.Index(["P1_d0", "P2_d0", "P3_d0", "P3_d7", "P4_d0", "P4_d7"], name="sample_id"),
)
n_cells = 600
obs = pd.DataFrame(
    {"sample_id": rng.choice(samples.index, n_cells), "cell_type": rng.choice(["B", "T", "Monocyte"], n_cells)},
    index=[f"cell_{i}" for i in range(n_cells)],
)
cells = AnnData(
    rng.poisson(1.0, (n_cells, 20)).astype(np.float32),
    obs=obs,
    var=pd.DataFrame(index=[f"gene_{i}" for i in range(20)]),
)
cells.obsm["X_pca"] = rng.normal(size=(n_cells, 5))

dd = DonorData(D=donors, S=samples, C=cells, donor_id="donor_id", sample_id="sample_id")
dd

╔═ DonorData(n_donors=4, n_samples=6, n_cells=600) ════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃                        ┃ S (samples, 1..2 per   ┃ C (cells, 88..114 per  ┃ ║
║ ┃ D (donors)             ┃ donor)                 ┃ sample)                ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ DataFrame with 4       │ DataFrame with 6       │ View of AnnData object │ ║
║ │ donors × 2 columns     │ samples × 2 columns    │ with n_obs × n_vars =  │ ║
║ │                        │                        │ 600 × 20               │ ║
║ │     columns: 'age',    │     columns:           │     obs: 'sample_id',  │ ║
║ │ 'disease'              │ 'donor_id', 'day'      │ 'cell_type'            │ ║
║ │                        │                        │     obsm: 'X_pca'      │ ║
║ └────────────────────────┴────────────────────────┴────────────────────────┘ ║
╚═ donor_id='donor_id', sample_id='sample_id' ═════════════════════════════════╝

## Levels

The levels are available as `D`, `S` and `C`. Tables are returned as DataFrames, AnnData and MuData levels as they
are.

In [3]:
dd.D

,age,disease
donor_id,,
P1,34,healthy
P2,52,healthy
P3,61,covid
P4,70,covid


In [4]:
dd.S

,donor_id,day
sample_id,,
P1_d0,P1,0
P2_d0,P2,0
P3_d0,P3,0
P3_d7,P3,7
P4_d0,P4,0
P4_d7,P4,7


In [5]:
dd.n_donors, dd.n_samples, dd.n_cells

(4, 6, 600)

Cells are stored grouped by sample, and samples grouped by donor, in the row order of the level above. The cells of a
unit are a contiguous block of `C`.

In [6]:
dd.cells_of("P3_d7", level="sample")

View of AnnData object with n_obs × n_vars = 99 × 20
    obs: 'sample_id', 'cell_type'
    obsm: 'X_pca'

## Selecting

A selection at one level selects the matching units at the other levels. Selecting donors keeps their samples and
cells, and selecting samples or cells drops the units above that are left without members.

In [7]:
def sizes(data):
    return {"donors": data.n_donors, "samples": data.n_samples, "cells": data.n_cells}


pd.DataFrame(
    {
        "all": sizes(dd),
        "covid donors": sizes(dd.filter_donors("disease == 'covid'")),
        "day 7 samples": sizes(dd.filter_samples("day == 7")),
        "B cells": sizes(dd.filter_cells("cell_type == 'B'")),
    }
)

,all,covid donors,day 7 samples,B cells
donors,4,2,2,4
samples,6,4,2,6
cells,600,412,213,198


## Moving information between levels

`get_df` collects variables from any level into a table at the requested level. Donor and sample columns are repeated
for their cells, and cell values are averaged for their samples or donors. Keys can be columns, genes or keys of
`obsm`.

In [8]:
dd.get_df(["age", "day", "cell_type", "gene_0"], level="cell").head()

,age,day,cell_type,gene_0
cell_5,34,0,B,0.0
cell_6,34,0,B,1.0
cell_7,34,0,Monocyte,2.0
cell_23,34,0,T,2.0
cell_27,34,0,T,0.0


In [9]:
dd.get_df(["disease", "gene_0", "X_pca"], level="sample")

,disease,gene_0,X_pca_0,X_pca_1,X_pca_2,X_pca_3,X_pca_4
sample_id,,,,,,,
P1_d0,healthy,1.050000,-0.068292,-0.049820,0.048927,-0.019036,0.030066
P2_d0,healthy,1.113636,0.146336,-0.136747,-0.074034,-0.000194,0.123579
P3_d0,covid,0.988889,0.062226,-0.109444,-0.131497,-0.053159,-0.137758
P3_d7,covid,1.121212,0.001638,-0.139523,-0.082912,-0.072922,0.155458
P4_d0,covid,1.018349,0.023926,0.128600,0.032593,0.008040,-0.253796
P4_d7,covid,0.929825,0.095759,0.204614,-0.084036,-0.003433,0.101666


Summaries of the cells, such as cell type compositions or pseudobulk profiles, are stored in `obsm` of their level.
They are subset together with the level.

In [10]:
dd.composition("cell_type", level="sample")
dd.aggregate(key_added="pseudobulk", level="sample")
dd.levels["sample"]

AnnData object with n_obs × n_vars = 6 × 0
    obs: 'donor_id', 'day'
    obsm: 'cell_type_fractions', 'pseudobulk'

In [11]:
covid = dd.filter_donors("disease == 'covid'")
covid.levels["sample"].obsm["cell_type_fractions"]

cell_type,B,Monocyte,T
sample_id,,,
P3_d0,0.277778,0.344444,0.377778
P3_d7,0.303030,0.323232,0.373737
P4_d0,0.339450,0.321101,0.339450
P4_d7,0.342105,0.333333,0.324561


## From and to a single AnnData

`to_anndata` attaches the donor and sample columns to every cell, and `from_anndata` splits such an AnnData into
levels again, moving the columns that are constant within donors or samples into their tables.

In [12]:
flat = dd.to_anndata()
DonorData.from_anndata(flat, donor_id="donor_id", sample_id="sample_id")

╔═ DonorData(n_donors=4, n_samples=6, n_cells=600) ════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃                        ┃ S (samples, 1..2 per   ┃ C (cells, 88..114 per  ┃ ║
║ ┃ D (donors)             ┃ donor)                 ┃ sample)                ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ DataFrame with 4       │ DataFrame with 6       │ AnnData object with    │ ║
║ │ donors × 2 columns     │ samples × 2 columns    │ n_obs × n_vars = 600 × │ ║
║ │                        │                        │ 20                     │ ║
║ │     columns: 'age',    │     columns:           │     obs: 'sample_id',  │ ║
║ │ 'disease'              │ 'donor_id', 'day'      │ 'cell_type',           │ ║
║ │                        │                        │ 'donor_id'             │ ║
║ │                        │                        │     obsm: 'X_pca'      │ ║
║ └────────────────────────┴────────────────────────┴────────────────────────┘ ║
╚═ donor_id='donor_id', sample_id='sample_id' ═════════════════════════════════╝

## Writing and reading

A `DonorData` is written to an HDF5 file or a Zarr store, see the {doc}`specification <../io/spec>`.

In [13]:
dd.write_dd("data/getting_started.dd.zarr")
donordata.io.read_dd("data/getting_started.dd.zarr")

╔═ DonorData(n_donors=4, n_samples=6, n_cells=600) ════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃                        ┃ S (samples, 1..2 per   ┃ C (cells, 88..114 per  ┃ ║
║ ┃ D (donors)             ┃ donor)                 ┃ sample)                ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ DataFrame with 4       │ DataFrame with 6       │ AnnData object with    │ ║
║ │ donors × 2 columns     │ samples × 2 columns    │ n_obs × n_vars = 600 × │ ║
║ │                        │                        │ 20                     │ ║
║ │     columns: 'age',    │     columns:           │     obs: 'sample_id',  │ ║
║ │ 'disease'              │ 'donor_id', 'day'      │ 'cell_type'            │ ║
║ │                        │     obsm:              │     obsm: 'X_pca'      │ ║
║ │                        │ 'pseudobulk',          │                        │ ║
║ │                        │ 'cell_type_fractions'  │                        │ ║
║ └────────────────────────┴────────────────────────┴────────────────────────┘ ║
╚═ donor_id='donor_id', sample_id='sample_id' ═════════════════════════════════╝